## **1. EarlyStopping 클래스 정의**

In [1]:
import torch
import numpy as np

class EarlyStopping:
    """주어진 patience 이후로 validation loss가 개선되지 않으면 학습을 조기 종료합니다."""
    def __init__(self, patience=7, verbose=False, delta=0, path='best_model.pth'):
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.Inf
        self.delta = delta
        self.path = path

    def __call__(self, val_loss, model):
        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping : {self.counter}/{self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        """Validation loss가 감소하면 모델을 저장합니다."""
        if self.verbose:
            print(f'Saving model : Val loss ({self.val_loss_min:.6f} --> {val_loss:.6f})')
        torch.save(model.state_dict(), self.path)
        self.val_loss_min = val_loss

## **2. 모델 학습 메서드 정의**

In [2]:
import wandb
from tqdm import tqdm

def train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    epochs,
    device,
    loss_fn=None,
    patience=5
):

    # 1. wandb 초기화 및 로깅 설정
    wandb.init(
        entity="team5pj1",
        project="health-eat-team5",
        config={
            "epochs": epochs,
            "patience": patience,
            "optimizer": optimizer.__class__.__name__,
            "learning_rate": optimizer.param_groups[0]['lr']
        }
    )

    model.to(device)
    early_stopping = EarlyStopping(patience=patience, verbose=True, path='best_pill_detector.pth')

    for epoch in range(epochs):
        # --- [Train Phase] ---
        model.train()
        train_loss_total = 0.0

        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
        for images, targets in train_pbar:
            images = list(image.to(device) for image in images)

            if isinstance(targets, list):
                targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            else:
                targets = targets.to(device)

            optimizer.zero_grad()

            if loss_fn is None:
                loss_dict = model(images, targets)
                losses = sum(loss for loss in loss_dict.values())
            else:
                outputs = model(images)
                losses = loss_fn(outputs, targets)

            losses.backward()
            optimizer.step()

            train_loss_total += losses.item()
            train_pbar.set_postfix({"loss": losses.item()})

        avg_train_loss = train_loss_total / len(train_loader)

        # --- [Validation Phase] ---
        val_loss_total = 0.0

        with torch.no_grad():
            val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]")
            for images, targets in val_pbar:
                images = list(image.to(device) for image in images)

                if isinstance(targets, list):
                    targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
                else:
                    targets = targets.to(device)

                # Validation Loss 계산
                if loss_fn is None:
                    model.train()
                    loss_dict = model(images, targets)
                    losses = sum(loss for loss in loss_dict.values())
                    model.eval()
                else:
                    outputs = model(images)
                    losses = loss_fn(outputs, targets)

                val_loss_total += losses.item()
                val_pbar.set_postfix({"loss": losses.item()})

        avg_val_loss = val_loss_total / len(val_loader)

        # 2. wandb에 결과 기록
        wandb.log({
            "epoch": epoch + 1,
            "train_loss": avg_train_loss,
            "val_loss": avg_val_loss
        })

        print(f"\n[Epoch {epoch+1}] Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

        # 3. Early Stopping 체크
        early_stopping(avg_val_loss, model)
        if early_stopping.early_stop:
            break

    # 학습 완료 시 wandb 세션 종료
    wandb.finish()

    # 최적의 모델 가중치 로드 후 반환
    model.load_state_dict(torch.load('best_model.pth'))
    return model

## **3. 학습 실행**

In [ ]:
import torch
import torch.optim as optim

# 하이퍼파라미터
NUM_CLASSES = 56 + 1
BATCH_SIZE = 8        # 객체 탐지는 메모리를 많이 차지하므로 우선 8 적용
EPOCHS = 50           # Early Stopping이 있으므로 넉넉하게 설정
LEARNING_RATE = 1e-4  # AdamW에 적합한 초기 학습률
WEIGHT_DECAY = 1e-4   # L2 정규화 효과
PATIENCE = 7          # 7 Epoch 동안 개선 없으면 조기 종료
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"사용 중인 디바이스: {DEVICE}")

# 모델, 옵티마이저 세팅
model = Model()

optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# 학습 실행
if __name__ == "__main__":
    best_model = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        epochs=EPOCHS,
        device=DEVICE,
        loss_fn=None,
        patience=PATIENCE,
    )